
# VoxCPM2 — Dual T4 Long-Form Burmese Voice Studio

This notebook is a lean replacement for the original single-GPU Voice Studio.

**What changes**
- Does **not** load a third copy of VoxCPM2 in the notebook process.
- Starts **two persistent worker processes**, one model on each T4.
- Splits long Burmese text at natural punctuation and sends chunks to both GPUs in parallel.
- Uses the **same saved reference voice** for every chunk.
- Saves permanent voices in a small voice library with a dropdown, play button, and delete button.
- Can restore the library automatically from an attached Kaggle Dataset.
- Can push the updated library back to a Kaggle Dataset with `kagglehub.dataset_upload`.

> Recommended: keep permanent reference clips short and clean (roughly one to three sentences). Long-form generation should be chunked rather than sent to VoxCPM2 as one giant request.


In [1]:
# Cell 1 — install only what is needed
!pip -q install -U voxcpm soundfile kagglehub huggingface_hub

print("✓ Runtime dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 59.0 MB/s eta 0:00:00
   

In [2]:
# Cell 2 — download model once to a shared local directory
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/kaggle/working/VoxCPM2"

if not os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")):
    print("Downloading VoxCPM2...")
    snapshot_download(
        repo_id="openbmb/VoxCPM2",
        local_dir=MODEL_DIR,
    )
else:
    print("✓ VoxCPM2 already exists in", MODEL_DIR)

print("✓ Model directory:", MODEL_DIR)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✓ Model directory: /kaggle/working/VoxCPM2


In [3]:
%%writefile /kaggle/working/voxcpm_dual_worker.py
import os
import traceback

def worker_main(physical_gpu_id, model_dir, task_q, result_q, optimize=True):
    # IMPORTANT: set this before importing torch / voxcpm in the child.
    os.environ["CUDA_VISIBLE_DEVICES"] = str(physical_gpu_id)

    try:
        import soundfile as sf
        from voxcpm import VoxCPM

        model = VoxCPM.from_pretrained(
            model_dir,
            load_denoiser=False,
            optimize=bool(optimize),
            device="cuda",
        )

        try:
            sample_rate = int(model.tts_model.sample_rate)
        except Exception:
            sample_rate = 48000

        result_q.put({
            "status": "ready",
            "gpu": physical_gpu_id,
            "sample_rate": sample_rate,
        })

    except Exception as e:
        result_q.put({
            "status": "startup_error",
            "gpu": physical_gpu_id,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        return

    while True:
        task = task_q.get()

        if task is None:
            break

        batch_id = task["batch_id"]
        task_id = task["task_id"]
        chunk_index = task.get("chunk_index", 0)
        output_path = task["output_path"]
        kwargs = task["kwargs"]

        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            wav = model.generate(**kwargs)

            sf.write(
                output_path,
                wav,
                sample_rate,
                subtype="PCM_16",
            )

            result_q.put({
                "status": "ok",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "output_path": output_path,
                "sample_rate": sample_rate,
                "samples": int(len(wav)),
                "duration": float(len(wav) / sample_rate),
            })

        except Exception as e:
            result_q.put({
                "status": "error",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })


Writing /kaggle/working/voxcpm_dual_worker.py


In [4]:
# Cell 4 — dual-GPU service + persistent voice library + UI
import os
import re
import sys
import json
import glob
import time
import uuid
import shutil
import queue as pyqueue
from pathlib import Path
from multiprocessing import get_context

import numpy as np
import soundfile as sf
import ipywidgets as widgets

from IPython.display import display, Audio, FileLink, clear_output

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

MODEL_DIR = "/kaggle/working/VoxCPM2"

# Change this only if your Kaggle dataset has a different owner/slug.
# This is used by the optional "Sync library to Kaggle Dataset" button.
VOICE_DATASET_HANDLE_DEFAULT = "cyawzin/voxcpm-voice-library"

# T4-safe default. Set True later only if you want to benchmark torch.compile.
# optimize=True can take a long time to compile/warm up on T4.
WORKER_OPTIMIZE = False

# Allow both child processes plenty of time to load the model.
WORKER_STARTUP_TIMEOUT = 900  # 15 minutes

LIBRARY_ROOT = Path("/kaggle/working/voxcpm_voice_library")
VOICE_DIR = LIBRARY_ROOT / "voices"
GENERATED_DIR = LIBRARY_ROOT / "generated"
META_FILE = LIBRARY_ROOT / "library.json"

VOICE_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RESTORE FROM AN ATTACHED KAGGLE DATASET
# ------------------------------------------------------------

def restore_attached_library_if_needed():
    if META_FILE.exists():
        return None

    candidates = glob.glob(
        "/kaggle/input/**/library.json",
        recursive=True,
    )

    for meta in candidates:
        src_root = Path(meta).parent
        src_voice_dir = src_root / "voices"

        if src_voice_dir.exists():
            shutil.copytree(
                src_root,
                LIBRARY_ROOT,
                dirs_exist_ok=True,
            )
            return str(src_root)

    return None

restored_from = restore_attached_library_if_needed()

# ------------------------------------------------------------
# LIBRARY HELPERS
# ------------------------------------------------------------

def empty_library():
    return {
        "version": 2,
        "active_voice_id": None,
        "voices": {},
    }

def load_library():
    if not META_FILE.exists():
        return empty_library()

    try:
        with open(META_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        data.setdefault("version", 2)
        data.setdefault("active_voice_id", None)
        data.setdefault("voices", {})
        return data
    except Exception:
        return empty_library()

library = load_library()

def save_library():
    LIBRARY_ROOT.mkdir(parents=True, exist_ok=True)
    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(
            library,
            f,
            ensure_ascii=False,
            indent=2,
        )

def voice_path(voice_id):
    entry = library["voices"][voice_id]
    return LIBRARY_ROOT / entry["file"]

def valid_voice_ids():
    out = []
    for voice_id, entry in library["voices"].items():
        p = LIBRARY_ROOT / entry["file"]
        if p.exists():
            out.append(voice_id)
    return out

# ------------------------------------------------------------
# LONG-TEXT CHUNKING
# ------------------------------------------------------------

BOUNDARIES = ["။", "!", "?", "…", ".", "၊", ",", ";", ":", " "]

def split_long_text(text, max_chars=220):
    """
    Split at natural Burmese/English boundaries.
    max_chars is a safety target, not an audio-duration guarantee.
    """
    text = re.sub(r"[ \t]+", " ", text or "")
    text = re.sub(r"\n+", "\n", text).strip()

    if not text:
        return []

    chunks = []
    remaining = text

    while len(remaining) > max_chars:
        window = remaining[:max_chars + 1]

        best = -1
        for mark in BOUNDARIES:
            pos = window.rfind(mark)
            if pos > best:
                best = pos

        # Avoid creating a tiny chunk just because there was old punctuation.
        if best < int(max_chars * 0.55):
            best = max_chars
        else:
            best += 1

        piece = remaining[:best].strip()
        if piece:
            chunks.append(piece)

        remaining = remaining[best:].strip()

    if remaining:
        chunks.append(remaining)

    return chunks

def concatenate_wavs(paths, output_path, pause_ms=90):
    arrays = []
    target_sr = None

    for i, p in enumerate(paths):
        wav, sr = sf.read(p, dtype="float32", always_2d=False)

        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        if target_sr is None:
            target_sr = sr
        elif sr != target_sr:
            raise RuntimeError(
                f"Sample-rate mismatch: expected {target_sr}, got {sr}"
            )

        arrays.append(wav)

        if i != len(paths) - 1 and pause_ms > 0:
            arrays.append(
                np.zeros(
                    int(target_sr * pause_ms / 1000.0),
                    dtype=np.float32,
                )
            )

    final = np.concatenate(arrays) if arrays else np.zeros(0, dtype=np.float32)

    sf.write(
        output_path,
        final,
        target_sr or 48000,
        subtype="PCM_16",
    )

    return output_path, target_sr or 48000, len(final)

# ------------------------------------------------------------
# TWO PERSISTENT GPU WORKERS
# ------------------------------------------------------------

sys.path.insert(0, "/kaggle/working")
from voxcpm_dual_worker import worker_main

class DualGPUVoxCPM:
    def __init__(
        self,
        model_dir,
        gpu_ids=(0, 1),
        optimize=False,
        startup_timeout=900,
        heartbeat_seconds=10,
    ):
        self.model_dir = model_dir
        self.gpu_ids = tuple(gpu_ids)
        self.ctx = get_context("spawn")
        self.task_q = self.ctx.Queue()
        self.result_q = self.ctx.Queue()
        self.procs = []
        self.sample_rate = 48000

        print(
            f"Starting {len(self.gpu_ids)} VoxCPM2 workers "
            f"(optimize={optimize})..."
        )

        for gpu_id in self.gpu_ids:
            p = self.ctx.Process(
                target=worker_main,
                args=(
                    gpu_id,
                    self.model_dir,
                    self.task_q,
                    self.result_q,
                    optimize,
                ),
                name=f"VoxCPM2-GPU-{gpu_id}",
            )
            p.start()
            self.procs.append(p)
            print(
                f"  • launched worker PID {p.pid} "
                f"for physical GPU {gpu_id}"
            )

        ready = set()
        deadline = time.time() + float(startup_timeout)
        last_status = 0.0

        while len(ready) < len(self.gpu_ids):
            now = time.time()

            for gpu_id, proc in zip(self.gpu_ids, self.procs):
                if gpu_id not in ready and proc.exitcode is not None:
                    self.shutdown(force=True)
                    raise RuntimeError(
                        f"VoxCPM2 worker for GPU {gpu_id} exited during startup "
                        f"with exit code {proc.exitcode}.\n"
                        "Scroll upward for the child-process traceback. "
                        "If this was an OOM kill, restart the Kaggle session "
                        "before running this cell again."
                    )

            if now > deadline:
                statuses = [
                    (
                        f"GPU {gpu}: pid={proc.pid}, "
                        f"alive={proc.is_alive()}, "
                        f"exitcode={proc.exitcode}, "
                        f"ready={gpu in ready}"
                    )
                    for gpu, proc in zip(self.gpu_ids, self.procs)
                ]
                self.shutdown(force=True)
                raise TimeoutError(
                    "Timed out while loading the VoxCPM2 GPU workers.\n"
                    + "\n".join(statuses)
                )

            # FIX: empty queue is normal while a large model is still loading.
            try:
                msg = self.result_q.get(timeout=5)
            except pyqueue.Empty:
                if now - last_status >= heartbeat_seconds:
                    waiting = [
                        str(gpu)
                        for gpu in self.gpu_ids
                        if gpu not in ready
                    ]
                    elapsed = int(
                        startup_timeout - max(0.0, deadline - now)
                    )
                    print(
                        f"  … still loading GPU worker(s) "
                        f"{', '.join(waiting)} "
                        f"({elapsed}s elapsed)"
                    )
                    last_status = now
                continue

            status = msg.get("status")

            if status == "ready":
                gpu = msg["gpu"]
                if gpu not in ready:
                    ready.add(gpu)
                    self.sample_rate = msg.get(
                        "sample_rate",
                        self.sample_rate,
                    )
                    print(
                        f"✓ GPU {gpu} worker ready "
                        f"({self.sample_rate} Hz)"
                    )

            elif status == "startup_error":
                self.shutdown(force=True)
                raise RuntimeError(
                    f"GPU {msg.get('gpu')} failed to start:\n"
                    f"{msg.get('error')}\n\n"
                    f"{msg.get('traceback', '')}"
                )

            else:
                print("Worker startup message:", msg)

        print("✓ All VoxCPM2 GPU workers are ready.")

    def run(self, task_specs):
        if not task_specs:
            return []

        batch_id = uuid.uuid4().hex
        wanted = set()

        for i, spec in enumerate(task_specs):
            task_id = f"{batch_id}_{i}"
            wanted.add(task_id)

            payload = dict(spec)
            payload["batch_id"] = batch_id
            payload["task_id"] = task_id
            self.task_q.put(payload)

        results = []

        while len(results) < len(task_specs):
            msg = self.result_q.get()

            if msg.get("batch_id") != batch_id:
                continue
            if msg.get("task_id") not in wanted:
                continue

            results.append(msg)

        errors = [r for r in results if r["status"] != "ok"]

        if errors:
            e = errors[0]
            raise RuntimeError(
                f"GPU {e.get('gpu')} generation failed:\n"
                f"{e.get('error')}\n\n"
                f"{e.get('traceback', '')}"
            )

        return sorted(
            results,
            key=lambda r: r.get("chunk_index", 0),
        )

    def shutdown(self, force=False):
        for _ in self.procs:
            try:
                self.task_q.put_nowait(None)
            except Exception:
                pass

        for p in self.procs:
            try:
                p.join(timeout=5)
            except Exception:
                pass

            if force or p.is_alive():
                try:
                    p.terminate()
                except Exception:
                    pass

# Shut down old workers if this cell is run again.
if "_VOX_SERVICE" in globals():
    try:
        _VOX_SERVICE.shutdown(force=True)
    except Exception:
        pass

print("Loading one VoxCPM2 replica on each T4...")

# IMPORTANT:
# If the previous broken cell already launched workers, restart the Kaggle
# session once before running this replacement cell so no orphan model remains.
#
# The notebook parent intentionally does NOT load VoxCPM2 itself.
# Only the two child workers own model replicas: one per physical T4.

_VOX_SERVICE = DualGPUVoxCPM(
    MODEL_DIR,
    gpu_ids=(0, 1),
    optimize=WORKER_OPTIMIZE,
    startup_timeout=WORKER_STARTUP_TIMEOUT,
    heartbeat_seconds=10,
)

print("✓ Dual-GPU VoxCPM2 service ready")

# ------------------------------------------------------------
# KAGGLE DATASET SYNC
# ------------------------------------------------------------

def sync_library_to_dataset(handle):
    handle = (handle or "").strip()

    if "/" not in handle:
        raise ValueError(
            "Dataset handle must look like: username/dataset-slug"
        )

    import kagglehub

    kagglehub.dataset_upload(
        handle,
        str(LIBRARY_ROOT),
        version_notes=time.strftime(
            "Voice library sync %Y-%m-%d %H:%M:%S"
        ),
    )

# ------------------------------------------------------------
# UI STATE
# ------------------------------------------------------------

STATE = {
    "candidate_path": None,
    "candidate_transcript": None,
    "candidate_direction": None,
}

title = widgets.HTML("""
<div style="
    background:#171717;
    color:white;
    padding:20px;
    border-radius:16px;
    margin-bottom:14px;
">
  <div style="font-size:25px;font-weight:700;">
    🎙️ VoxCPM2 Dual-T4 Burmese Voice Studio
  </div>
  <div style="margin-top:8px;color:#d7d7d7;line-height:1.55;">
    Long scripts are split into short natural chunks and generated
    on both T4 GPUs with the same saved voice.
  </div>
</div>
""")

# ---------- permanent voice row ----------

saved_voice = widgets.Dropdown(
    description="Permanent voice:",
    layout=widgets.Layout(width="470px"),
)

play_voice_btn = widgets.Button(
    description="▶ Play",
    layout=widgets.Layout(width="90px"),
)

delete_voice_btn = widgets.Button(
    description="🗑 Delete",
    button_style="danger",
    layout=widgets.Layout(width="105px"),
)

dataset_handle = widgets.Text(
    value=VOICE_DATASET_HANDLE_DEFAULT,
    description="Dataset:",
    placeholder="username/voxcpm-voice-library",
    layout=widgets.Layout(width="470px"),
)

auto_sync = widgets.Checkbox(
    value=False,
    description="Auto-sync after Save/Delete",
    indent=False,
)

sync_btn = widgets.Button(
    description="☁ Sync library to Kaggle Dataset",
    button_style="info",
    layout=widgets.Layout(width="280px"),
)

library_output = widgets.Output()

def refresh_saved_voices(select_id=None):
    ids = valid_voice_ids()

    ids.sort(
        key=lambda vid: library["voices"][vid].get("created", ""),
        reverse=True,
    )

    if ids:
        options = [
            (
                f"{library['voices'][vid].get('name', vid)}"
                f"  •  {library['voices'][vid].get('created', '')}",
                vid,
            )
            for vid in ids
        ]

        saved_voice.options = options

        desired = (
            select_id
            or library.get("active_voice_id")
            or ids[0]
        )

        saved_voice.value = desired if desired in ids else ids[0]
        delete_voice_btn.disabled = False
        play_voice_btn.disabled = False

    else:
        saved_voice.options = [
            ("— no permanent voices saved yet —", "")
        ]
        saved_voice.value = ""
        delete_voice_btn.disabled = True
        play_voice_btn.disabled = True

def selected_voice_id():
    return saved_voice.value or ""

def set_active_voice(change=None):
    vid = selected_voice_id()
    if vid and vid in library["voices"]:
        library["active_voice_id"] = vid
        save_library()

saved_voice.observe(set_active_voice, names="value")

def on_play_voice(_):
    with library_output:
        clear_output()
        vid = selected_voice_id()

        if not vid:
            print("No permanent voice selected.")
            return

        p = voice_path(vid)
        print(library["voices"][vid]["name"])
        display(Audio(str(p), autoplay=False))

def maybe_sync():
    if not auto_sync.value:
        return

    sync_library_to_dataset(dataset_handle.value)

def on_delete_voice(_):
    with library_output:
        clear_output()

        vid = selected_voice_id()
        if not vid:
            return

        entry = library["voices"].get(vid, {})
        p = LIBRARY_ROOT / entry.get("file", "")

        if p.exists():
            p.unlink()

        library["voices"].pop(vid, None)

        if library.get("active_voice_id") == vid:
            library["active_voice_id"] = None

        save_library()
        refresh_saved_voices()

        print("✓ Voice deleted from the working library.")

        if auto_sync.value:
            try:
                maybe_sync()
                print("✓ Dataset updated; the deletion is persistent.")
            except Exception as e:
                print("⚠ Local deletion succeeded, but dataset sync failed:")
                print(e)

def on_sync(_):
    with library_output:
        clear_output()

        try:
            print("Uploading current voice library...")
            sync_library_to_dataset(dataset_handle.value)
            print("✓ Kaggle Dataset synced.")
            print(
                "On a fresh notebook session, attach/use the latest "
                "version of this dataset as an Input."
            )
        except Exception as e:
            print("❌ Dataset sync failed:")
            print(e)

play_voice_btn.on_click(on_play_voice)
delete_voice_btn.on_click(on_delete_voice)
sync_btn.on_click(on_sync)

# ---------- audition / save a new permanent voice ----------

voice_name = widgets.Text(
    value="My Burmese Voice",
    description="Save name:",
    layout=widgets.Layout(width="470px"),
)

voice_direction = widgets.Textarea(
    value=(
        "A natural adult Burmese speaker with a warm, clear, conversational voice. "
        "Use authentic Burmese rhythm, clean articulation, realistic pauses, "
        "and human prosody. Avoid robotic or exaggerated announcer delivery."
    ),
    description="",
    placeholder="Describe the voice you want...",
    layout=widgets.Layout(width="100%", height="110px"),
)

audition_text = widgets.Textarea(
    value=(
        "မင်္ဂလာပါ။ ဒီအသံက သဘာဝကျပြီး နားထောင်ရလွယ်ကူတဲ့ "
        "မြန်မာစကားပြောသံ ဖြစ်စေချင်ပါတယ်။"
    ),
    description="",
    placeholder="Use one to three sentences for auditioning...",
    layout=widgets.Layout(width="100%", height="110px"),
)

audition_btn = widgets.Button(
    description="🎲 Generate Voice Candidate",
    button_style="success",
    layout=widgets.Layout(width="245px", height="45px"),
)

save_candidate_btn = widgets.Button(
    description="🔒 Save as Permanent Voice",
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(width="245px", height="45px"),
)

audition_output = widgets.Output()

def on_audition(_):
    with audition_output:
        clear_output()

        text = audition_text.value.strip()
        direction = voice_direction.value.strip()

        if not text:
            print("❌ Enter a short audition script.")
            return

        audition_btn.disabled = True
        save_candidate_btn.disabled = True

        try:
            designed_text = (
                f"({direction}){text}"
                if direction
                else text
            )

            out = GENERATED_DIR / (
                f"candidate_{int(time.time())}_{uuid.uuid4().hex[:6]}.wav"
            )

            print("Generating a fresh candidate on one available T4...")

            results = _VOX_SERVICE.run([
                {
                    "chunk_index": 0,
                    "output_path": str(out),
                    "kwargs": {
                        "text": designed_text,
                        "cfg_value": 2.0,
                        "inference_timesteps": 10,
                        "retry_badcase": True,
                    },
                }
            ])

            STATE["candidate_path"] = str(out)
            STATE["candidate_transcript"] = text
            STATE["candidate_direction"] = direction

            r = results[0]

            print(
                f"✓ Candidate ready in {r['duration']:.1f}s of audio "
                f"(GPU {r['gpu']})"
            )

            display(Audio(str(out), autoplay=False))
            save_candidate_btn.disabled = False

        except Exception as e:
            print("❌ Candidate generation failed:")
            print(e)

        finally:
            audition_btn.disabled = False

def on_save_candidate(_):
    with audition_output:
        clear_output()

        candidate = STATE.get("candidate_path")

        if not candidate or not os.path.exists(candidate):
            print("❌ Generate a candidate first.")
            return

        name = voice_name.value.strip() or "Permanent Voice"
        vid = uuid.uuid4().hex[:12]
        dest = VOICE_DIR / f"{vid}.wav"

        shutil.copy2(candidate, dest)

        library["voices"][vid] = {
            "name": name,
            "file": f"voices/{vid}.wav",
            "transcript": STATE.get("candidate_transcript") or "",
            "direction": STATE.get("candidate_direction") or "",
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

        library["active_voice_id"] = vid
        save_library()
        refresh_saved_voices(select_id=vid)

        print(f"🔒 Saved permanent voice: {name}")
        display(Audio(str(dest), autoplay=False))

        if auto_sync.value:
            try:
                print("Syncing to Kaggle Dataset...")
                maybe_sync()
                print("✓ Permanent copy synced.")
            except Exception as e:
                print("⚠ Voice is saved locally, but dataset sync failed:")
                print(e)

audition_btn.on_click(on_audition)
save_candidate_btn.on_click(on_save_candidate)

# ---------- long script ----------

long_script = widgets.Textarea(
    value="",
    description="",
    placeholder="Paste your long Burmese script here...",
    layout=widgets.Layout(width="100%", height="270px"),
)

chunk_chars = widgets.IntSlider(
    value=220,
    min=120,
    max=360,
    step=10,
    description="Chunk chars:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

pause_ms = widgets.IntSlider(
    value=90,
    min=0,
    max=300,
    step=10,
    description="Join pause:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

run_style = widgets.Text(
    value="",
    description="Style:",
    placeholder="optional: slightly faster, energetic, calm, etc.",
    layout=widgets.Layout(width="100%"),
)

generate_long_btn = widgets.Button(
    description="⚡ Generate Long Voiceover on 2 GPUs",
    button_style="success",
    layout=widgets.Layout(width="330px", height="50px"),
)

generation_output = widgets.Output()

def on_generate_long(_):
    with generation_output:
        clear_output()

        vid = selected_voice_id()

        if not vid:
            print("❌ Save/select a permanent voice first.")
            return

        script = long_script.value.strip()

        if not script:
            print("❌ Paste a script first.")
            return

        entry = library["voices"][vid]
        ref = voice_path(vid)
        transcript = entry.get("transcript", "").strip()

        if not ref.exists():
            print("❌ Saved reference WAV is missing.")
            return

        if not transcript:
            print(
                "❌ This saved voice has no reference transcript. "
                "Generate and save a new permanent voice."
            )
            return

        chunks = split_long_text(
            script,
            max_chars=int(chunk_chars.value),
        )

        if not chunks:
            print("❌ No text chunks were created.")
            return

        print(
            f"Voice: {entry.get('name', vid)}\n"
            f"Chunks: {len(chunks)}\n"
            f"Two workers will pull chunks from the same queue."
        )
        print()

        generate_long_btn.disabled = True

        batch_stamp = time.strftime("%Y%m%d_%H%M%S")
        chunk_dir = GENERATED_DIR / f"chunks_{batch_stamp}_{uuid.uuid4().hex[:5]}"
        chunk_dir.mkdir(parents=True, exist_ok=True)

        style = run_style.value.strip()
        tasks = []

        for i, chunk in enumerate(chunks):
            target_text = f"({style}){chunk}" if style else chunk
            out = chunk_dir / f"chunk_{i:04d}.wav"

            tasks.append({
                "chunk_index": i,
                "output_path": str(out),
                "kwargs": {
                    "text": target_text,

                    # Same permanent voice on EVERY chunk.
                    "reference_wav_path": str(ref),
                    "prompt_wav_path": str(ref),
                    "prompt_text": transcript,

                    "cfg_value": 2.0,
                    "inference_timesteps": 10,
                    "retry_badcase": True,
                },
            })

        start = time.time()

        try:
            results = _VOX_SERVICE.run(tasks)

            ordered_paths = [
                r["output_path"]
                for r in results
            ]

            final_path = GENERATED_DIR / (
                f"long_voiceover_{batch_stamp}.wav"
            )

            _, sr, samples = concatenate_wavs(
                ordered_paths,
                str(final_path),
                pause_ms=int(pause_ms.value),
            )

            elapsed = time.time() - start
            duration = samples / sr
            used_gpus = sorted({r["gpu"] for r in results})

            print()
            print(f"✓ Finished in {elapsed:.1f}s")
            print(f"✓ Final audio duration: {duration:.1f}s")
            print(f"✓ GPUs used: {used_gpus}")
            print(f"✓ Output: {final_path}")
            print()

            display(Audio(str(final_path), autoplay=False))
            display(
                FileLink(
                    str(final_path),
                    result_html_prefix="📥 WAV: ",
                )
            )

        except Exception as e:
            print("❌ Long-form generation failed:")
            print(e)

        finally:
            generate_long_btn.disabled = False

generate_long_btn.on_click(on_generate_long)

# ------------------------------------------------------------
# INITIALIZE + DISPLAY
# ------------------------------------------------------------

refresh_saved_voices()

if restored_from:
    with library_output:
        print("✓ Restored saved voices from attached Dataset:")
        print(restored_from)

permanent_row = widgets.HBox(
    [saved_voice, play_voice_btn, delete_voice_btn],
    layout=widgets.Layout(gap="8px", flex_flow="row wrap"),
)

sync_row = widgets.HBox(
    [dataset_handle, auto_sync, sync_btn],
    layout=widgets.Layout(gap="10px", flex_flow="row wrap"),
)

audition_buttons = widgets.HBox(
    [audition_btn, save_candidate_btn],
    layout=widgets.Layout(gap="10px", flex_flow="row wrap"),
)

display(title)

display(widgets.HTML("<h3>1. Permanent Voice Library</h3>"))
display(permanent_row)
display(sync_row)
display(library_output)

display(widgets.HTML("<h3>2. Create / Save a New Voice</h3>"))
display(widgets.HTML("<b>Voice direction</b>"))
display(voice_direction)
display(widgets.HTML("<b>Short audition text (1–3 sentences)</b>"))
display(audition_text)
display(voice_name)
display(audition_buttons)
display(audition_output)

display(widgets.HTML("<h3>3. Long Script → Dual-GPU Voiceover</h3>"))
display(long_script)
display(run_style)
display(chunk_chars)
display(pause_ms)
display(generate_long_btn)
display(generation_output)


Loading one VoxCPM2 replica on each T4...
Starting 2 VoxCPM2 workers (optimize=False)...
  • launched worker PID 145 for physical GPU 0
  • launched worker PID 146 for physical GPU 1
  … still loading GPU worker(s) 0, 1 (0s elapsed)
  … still loading GPU worker(s) 0, 1 (10s elapsed)


voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Running on device: cuda, dtype: bfloat16
Running on device: cuda, dtype: bfloat16


  … still loading GPU worker(s) 0, 1 (20s elapsed)
  … still loading GPU worker(s) 0, 1 (30s elapsed)


Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors
Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors


  … still loading GPU worker(s) 0, 1 (40s elapsed)


...

Text(value='', description='Style:', layout=Layout(width='100%'), placeholder='optional: slightly faster, ener…

IntSlider(value=220, continuous_update=False, description='Chunk chars:', layout=Layout(width='500px'), max=36…

IntSlider(value=90, continuous_update=False, description='Join pause:', layout=Layout(width='500px'), max=300,…

Button(button_style='success', description='⚡ Generate Long Voiceover on 2 GPUs', layout=Layout(height='50px',…

Output()


## Persistence notes

- `/kaggle/working` is suitable for the current running environment, but it should **not** be your only permanent copy for future Kaggle sessions.
- An attached Kaggle Dataset is read-only under `/kaggle/input`.
- The **Sync library to Kaggle Dataset** button uploads the current working library as a new Dataset version.
- On a fresh session, attach that Dataset as an Input. The UI automatically searches `/kaggle/input/**/library.json` and restores the saved voices.
- Deleting a voice and syncing again publishes a new Dataset version without that voice.

For best long-form quality, keep the chunk size around **180–260 characters** to start, then adjust for your narration speed.
